In [4]:
#starting off and setting up
import pandas as pd
import numpy as np
from scipy.optimize import minimize

prices = pd.read_csv("prices.csv", index_col=0, parse_dates=True)
returns = prices.pct_change().dropna()
n_assets = len(returns.columns)

bounds = tuple((0, 1) for _ in range(n_assets))
initial = np.repeat(1/n_assets, n_assets)

In [5]:
#split the 10-year data into sample and testing ground
split_date = "2020-01-01"
sample = returns.loc[:split_date]
test=returns.loc[split_date:]
print(test.head())

                AAPL       CAT       JNJ       JPM        KO      MSFT  \
Date                                                                     
2020-01-02  0.022817  0.019299  0.000686  0.012124 -0.006504  0.018516   
2020-01-03 -0.009722 -0.013884 -0.011578 -0.013196 -0.005456 -0.012451   
2020-01-06  0.007968 -0.000674 -0.001248 -0.000795 -0.000366  0.002585   
2020-01-07 -0.004703 -0.013213  0.006107 -0.017001 -0.007682 -0.009118   
2020-01-08  0.016086  0.008881 -0.000138  0.007801  0.001843  0.015928   

                NVDA        PG       WMT       XOM  
Date                                                
2020-01-02  0.019592 -0.011930  0.000842  0.016050  
2020-01-03 -0.016006 -0.006725 -0.008828 -0.008039  
2020-01-06  0.004194  0.001387 -0.002036  0.007678  
2020-01-07  0.012106 -0.006192 -0.009265 -0.008184  
2020-01-08  0.001876  0.004263 -0.003432 -0.015080  


In [6]:
#get mean returns, covariances and set risk-free rate
sample_m = sample.mean() * 252
sample_cov = sample.cov() * 252
rfr = 0.02

#define function to get Sharpe ratio from given portfolio and maximise
def neg_sharpe_sample(w):
    r = np.sum(w * sample_m)
    v = np.sqrt(w.T @ sample_cov @ w)
    return -(r - rfr)/v

sum_to_one = {'type': 'eq', 'fun': lambda w: np.sum(w)-1}
sample_optimise = minimize(neg_sharpe_sample, initial, bounds=bounds, constraints=sum_to_one)
sample_w = sample_optimise.x

#get max Sharpe ratio and the portfolio producing it
print(-sample_optimise.fun)
print(pd.Series(sample_w, index=returns.columns).round(4))

1.497731739805984
AAPL    0.0325
CAT     0.0000
JNJ     0.0000
JPM     0.1612
KO      0.0849
MSFT    0.3068
NVDA    0.3196
PG      0.0775
WMT     0.0175
XOM     0.0000
dtype: float64


In [7]:
#now apply this optimised portfolio out-of-sample
oos_returns = test @ sample_w
print(oos_returns)
oos_ann_return = oos_returns.mean() *252
oos_ann_vol = oos_returns.std() * np.sqrt(252)
oos_sharpe = (oos_ann_return-rfr)/oos_ann_vol
print(oos_ann_return, oos_ann_vol, oos_sharpe)

Date
2020-01-02    0.013175
2020-01-03   -0.012518
2020-01-06    0.002305
2020-01-07   -0.003115
2020-01-08    0.007693
                ...   
2024-12-23    0.010915
2024-12-24    0.008617
2024-12-26   -0.000642
2024-12-27   -0.014376
2024-12-30   -0.006504
Length: 1257, dtype: float64
0.38506536974824274 0.29403873190011304 1.2415553807797604


In [8]:
#comapre against 1/n (naive)
equal_w = np.repeat(1/n_assets, n_assets)
ew_returns = test @ equal_w
ew_ann_return = ew_returns.mean() * 252
ew_ann_vol = ew_returns.std() * np.sqrt(252)
ew_sharpe = (ew_ann_return-rfr)/ew_ann_vol
print(ew_ann_return, ew_ann_vol, ew_sharpe)

0.24021938461574607 0.20280701183235694 1.0858568578377479


In [9]:
#now do oos testing across multiple timeframes
split_dates = ["2018-01-01", "2019-01-01", "2020-01-01", "2021-01-01", "2022-01-01"]
results=[]

train_years = 3
test_years = 1

start_year = returns.index[0].year
end_year = returns.index[-1].year

def neg_sharpe_sample(w, mu, sig):
    r = np.sum(w * mu)
    v = np.sqrt(w.T @ sig @ w)
    return -(r - rfr)/v

for year_split in range(start_year + train_years, end_year + 1):
    train_start = f"{year_split - train_years}-01-01"
    train_end = f"{year_split}-01-01"
    test_end = f"{year_split + test_years}-01-01"
    print(train_start, train_end, test_end)
    
    sample = returns.loc[train_start:train_end]
    test = returns.loc[train_end:test_end]

    sample_m=sample.mean()*252
    sample_cov=sample.cov()*252
    sample_optimise = minimize(neg_sharpe_sample, initial, args=(sample_m, sample_cov), bounds=bounds, constraints=sum_to_one)
    opt_w = sample_optimise.x
    OOS_daily_rets = test @ opt_w
    OOS_sharpe = (OOS_daily_rets.mean()*252 - rfr) / (OOS_daily_rets.std()*np.sqrt(252))
    EW_daily_rets = test @ equal_w
    EW_sharpe = (EW_daily_rets.mean()*252 - rfr) / (EW_daily_rets.std()*np.sqrt(252))

    results.append({
    'Test Year': year_split,
    'In-sample Sharpe': -sample_optimise.fun,
    'OOS Sharpe': OOS_sharpe,
    'Equal-weighted Sharpe': EW_sharpe,
    'Max Weight': opt_w.max()
        })
DF_results=pd.DataFrame(results)
DF_results.round(3)

2015-01-01 2018-01-01 2019-01-01
2016-01-01 2019-01-01 2020-01-01
2017-01-01 2020-01-01 2021-01-01
2018-01-01 2021-01-01 2022-01-01
2019-01-01 2022-01-01 2023-01-01
2020-01-01 2023-01-01 2024-01-01
2021-01-01 2024-01-01 2025-01-01


,Test Year,In-sample Sharpe,OOS Sharpe,Equal-weighted Sharpe,Max Weight
0,2018,2.119,-0.556,-0.251,0.494
1,2019,1.480,2.019,2.359,0.321
2,2020,1.800,1.034,0.800,0.310
3,2021,1.276,1.617,2.263,0.599
4,2022,1.831,-0.830,-0.085,0.419
5,2023,0.928,2.413,2.142,0.392
6,2024,1.536,2.149,2.863,0.658


In [10]:
DF_results['MVO beats EW?']=DF_results['OOS Sharpe']>DF_results['Equal-weighted Sharpe']
DF_results.round(3)

,Test Year,In-sample Sharpe,OOS Sharpe,Equal-weighted Sharpe,Max Weight,MVO beats EW?
0,2018,2.119,-0.556,-0.251,0.494,False
1,2019,1.480,2.019,2.359,0.321,False
2,2020,1.800,1.034,0.800,0.310,True
3,2021,1.276,1.617,2.263,0.599,False
4,2022,1.831,-0.830,-0.085,0.419,False
5,2023,0.928,2.413,2.142,0.392,True
6,2024,1.536,2.149,2.863,0.658,False


In [11]:
print(DF_results['OOS Sharpe'].mean())
print(DF_results['Equal-weighted Sharpe'].mean())

1.1209706386001235
1.4416298314464018


In [12]:
from sklearn.covariance import LedoitWolf

In [17]:
#now do ledoit-Wolf shrinkage
split_dates = ["2018-01-01", "2019-01-01", "2020-01-01", "2021-01-01", "2022-01-01"]
results=[]

train_years = 3
test_years = 1

start_year = returns.index[0].year
end_year = returns.index[-1].year

def neg_sharpe_sample(w, mu, sig):
    r = np.sum(w * mu)
    v = np.sqrt(w.T @ sig @ w)
    return -(r - rfr)/v

sample_weights_history = []
shrunk_weights_history = []

for year_split in range(start_year + train_years, end_year + 1):
    train_start = f"{year_split - train_years}-01-01"
    train_end = f"{year_split}-01-01"
    test_end = f"{year_split + test_years}-01-01"
    print(train_start, train_end, test_end)
    
    sample = returns.loc[train_start:train_end]
    test = returns.loc[train_end:test_end]

    sample_m=sample.mean()*252
    
    sample_cov=sample.cov()*252
    lw_cov = LedoitWolf().fit(sample).covariance_*252
    
    sample_optimise = minimize(neg_sharpe_sample, initial, args=(sample_m, sample_cov), bounds=bounds, constraints=sum_to_one)
    opt_w = sample_optimise.x
    OOS_daily_rets = test @ opt_w
    OOS_sharpe = (OOS_daily_rets.mean()*252 - rfr) / (OOS_daily_rets.std()*np.sqrt(252))
    
    lw_optimise = minimize(neg_sharpe_sample, initial, args=(sample_m, lw_cov), bounds=bounds, constraints=sum_to_one)
    lw_w=lw_optimise.x
    lw_daily_rets = test @ lw_w
    lw_sharpe = (lw_daily_rets.mean()*252 - rfr) / (lw_daily_rets.std()*np.sqrt(252))
    
    EW_daily_rets = test @ equal_w
    EW_sharpe = (EW_daily_rets.mean()*252 - rfr) / (EW_daily_rets.std()*np.sqrt(252))

    sample_weights_history.append(sample_optimise.x)
    shrunk_weights_history.append(lw_optimise.x)
    
    results.append({
        'Test Year': year_split,
        'In-sample Sharpe': -sample_optimise.fun,
        'Equal-weighted Sharpe': EW_sharpe,
        'MVO Sharpe': OOS_sharpe,
        'Shrunk Sharpe': lw_sharpe,
        'MVO Max Weight': opt_w.max(),
        'Shrunk Max Weight': lw_w.max(),
        'Shrinkage': LedoitWolf().fit(sample).shrinkage_,
        "Sample Vol": (test @ sample_optimise.x).std() * np.sqrt(252),
        "Shrunk Vol": (test @ lw_optimise.x).std() * np.sqrt(252),
        "EW Vol": (test @ equal_w).std() * np.sqrt(252),
    })

def turnover(history):
    arr = np.array(history)
    changes = np.abs(np.diff(arr, axis=0)).sum(axis=1)
    return changes.mean()

print(turnover(sample_weights_history))
print(turnover(shrunk_weights_history))

DF_results=pd.DataFrame(results)
DF_results.round(3)

2015-01-01 2018-01-01 2019-01-01
2016-01-01 2019-01-01 2020-01-01
2017-01-01 2020-01-01 2021-01-01
2018-01-01 2021-01-01 2022-01-01
2019-01-01 2022-01-01 2023-01-01
2020-01-01 2023-01-01 2024-01-01
2021-01-01 2024-01-01 2025-01-01
0.9642889952405156
0.9478917340438233


,Test Year,In-sample Sharpe,Equal-weighted Sharpe,MVO Sharpe,Shrunk Sharpe,MVO Max Weight,Shrunk Max Weight,Shrinkage,Sample Vol,Shrunk Vol,EW Vol
0,2018,2.119,-0.251,-0.556,-0.544,0.494,0.499,0.045,0.309,0.313,0.183
1,2019,1.480,2.359,2.019,2.028,0.321,0.309,0.036,0.180,0.180,0.140
2,2020,1.800,0.800,1.034,1.032,0.310,0.305,0.021,0.335,0.335,0.340
3,2021,1.276,2.263,1.617,1.629,0.599,0.581,0.030,0.199,0.197,0.128
4,2022,1.831,-0.085,-0.830,-0.830,0.419,0.424,0.035,0.403,0.404,0.219
5,2023,0.928,2.142,2.413,2.419,0.392,0.388,0.025,0.228,0.227,0.122
6,2024,1.536,2.863,2.149,2.154,0.658,0.655,0.015,0.198,0.199,0.103


In [22]:
DF_results['Sharpe increase']=DF_results.apply(lambda x: ((x['Shrunk Sharpe']-x['MVO Sharpe'])/x['MVO Sharpe'])*100 if x['MVO Sharpe']>0 else ((x['Shrunk Sharpe']-x['MVO Sharpe'])/x['MVO Sharpe'])*100*-1, axis=1)
DF_results.round(3)

,Test Year,In-sample Sharpe,Equal-weighted Sharpe,MVO Sharpe,Shrunk Sharpe,MVO Max Weight,Shrunk Max Weight,Shrinkage,Sample Vol,Shrunk Vol,EW Vol,Sharpe increase
0,2018,2.119,-0.251,-0.556,-0.544,0.494,0.499,0.045,0.309,0.313,0.183,2.172
1,2019,1.480,2.359,2.019,2.028,0.321,0.309,0.036,0.180,0.180,0.140,0.439
2,2020,1.800,0.800,1.034,1.032,0.310,0.305,0.021,0.335,0.335,0.340,-0.261
3,2021,1.276,2.263,1.617,1.629,0.599,0.581,0.030,0.199,0.197,0.128,0.730
4,2022,1.831,-0.085,-0.830,-0.830,0.419,0.424,0.035,0.403,0.404,0.219,-0.039
5,2023,0.928,2.142,2.413,2.419,0.392,0.388,0.025,0.228,0.227,0.122,0.273
6,2024,1.536,2.863,2.149,2.154,0.658,0.655,0.015,0.198,0.199,0.103,0.239


In [23]:
DF_results['Sharpe increase'].mean()

np.float64(0.5075148430751202)

In [24]:
DF_results['Shrunk Sharpe'].mean()

np.float64(1.126891741812441)